# RuBERT + NLI FactCheck: гибридная система с реальной проверкой фактов

## Архитектура

```
Входной текст
      │
      ├──► RuBERT → P(fake), P(real)            [существующая модель]
      │
      └──► RealFactChecker
               │
               ├─► Natasha NER → сущности
               │
               ├─► Evidence Retrieval
               │       ├── Wikipedia API (ru)    [бесплатно, не заблокирован]
               │       └── DuckDuckGo Search     [бесплатно, не заблокирован]
               │
               ├─► SentenceTransformer → топ-K релевантных доказательств
               │       (ищем самые близкие предложения из источников)
               │
               └─► RuBERT-NLI (cointegrated/rubert-base-cased-nli-threeway)
                       premise   = доказательство из Wikipedia/DDG
                       hypothesis = утверждение из статьи
                       → P(entailment) - P(contradiction) ∈ [-1, 1]

factcheck_score = 0.5 + 0.5 × mean(nli_scores)  ∈ [0, 1]
final_real = α × RuBERT_real + (1-α) × factcheck_score
```

Источники, доступные в России: **ru.wikipedia.org** и **DuckDuckGo** (не заблокированы).


## 1. Установка зависимостей

In [1]:
!pip install sentence-transformers duckduckgo-search -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Импорты

In [2]:
import os
import re
import json
import time
import random
import hashlib
import warnings
warnings.filterwarnings('ignore')
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report
)
from sklearn.metrics.pairwise import cosine_similarity as cos_sim_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from tqdm.auto import tqdm

# Natasha — русский NER
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, Doc

# Wikipedia
import wikipedia
wikipedia.set_lang('ru')

# DuckDuckGo Search — поиск без API-ключа
from duckduckgo_search import DDGS

# Sentence embeddings для retrieval
from sentence_transformers import SentenceTransformer

print('Все импорты успешны')


Все импорты успешны


## 3. Конфигурация

In [3]:
@dataclass
class Config:
    # Данные
    data_path:      str   = 'data/ready_dataset.csv'
    text_column:    str   = 'combined_text'
    label_column:   str   = 'label'

    # RuBERT (классификатор fake/real)
    model_name:     str   = 'DeepPavlov/rubert-base-cased'
    output_dir:     str   = 'models/rubert'         # где лежит обученная модель

    # Обучение
    max_length:     int   = 256
    batch_size:     int   = 8
    epochs:         int   = 4
    learning_rate:  float = 2e-5
    weight_decay:   float = 0.01
    warmup_ratio:   float = 0.1
    grad_clip:      float = 1.0

    # Сплит
    test_size:      float = 0.1
    val_size:       float = 0.1
    random_state:   int   = 42
    num_workers:    int   = 0

    # NLI модель (проверка фактов)
    nli_model_name: str   = 'cointegrated/rubert-base-cased-nli-threeway'

    # Retrieval
    retrieval_model: str  = 'paraphrase-multilingual-mpnet-base-v2'
    fc_cache_dir:    str  = 'fact_check_cache'
    fc_max_entities: int  = 4          # сущностей из текста
    fc_top_k_evidence: int = 3         # топ-K доказательств для NLI
    fc_max_wiki_chars: int = 5000      # макс символов из Wikipedia
    fc_max_article_sents: int = 6      # предложений статьи для проверки
    fc_ddg_results:  int  = 3          # результатов из DDG

    # Гибридный детектор
    alpha:           float = 0.7       # вес RuBERT; (1-alpha) = вес factcheck
    threshold:       float = 0.5

    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def save_json(path, data):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

print('Config готов')


Config готов


## 4. Обучение RuBERT

> Блок идентичен `train_rubert.ipynb`. Пропустите, если модель уже обучена.

In [ ]:
def load_dataset(config: Config):
    df = pd.read_csv(config.data_path)
    for col in [config.text_column, config.label_column]:
        if col not in df.columns:
            raise ValueError(f'Колонка {col} не найдена. Есть: {list(df.columns)}')
    df = df[[config.text_column, config.label_column]].dropna()
    df[config.text_column] = df[config.text_column].astype(str).str.strip()
    df = df[df[config.text_column] != '']
    df[config.label_column] = pd.to_numeric(df[config.label_column], errors='coerce')
    df = df.dropna(subset=[config.label_column])
    df[config.label_column] = df[config.label_column].astype(int)
    train_val, test = train_test_split(
        df, test_size=config.test_size, random_state=config.random_state,
        stratify=df[config.label_column])
    rel_val = config.val_size / (1 - config.test_size)
    train, val = train_test_split(
        train_val, test_size=rel_val, random_state=config.random_state,
        stratify=train_val[config.label_column])
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


class NewsDataset(Dataset):
    def __init__(self, df, tokenizer, text_col, label_col, max_length):
        self.texts  = df[text_col].tolist()
        self.labels = df[label_col].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)}


def create_dataloaders(train_df, val_df, test_df, tokenizer, config):
    mk = lambda df, shuf: DataLoader(
        NewsDataset(df, tokenizer, config.text_column, config.label_column, config.max_length),
        batch_size=config.batch_size, shuffle=shuf, num_workers=config.num_workers)
    return mk(train_df, True), mk(val_df, False), mk(test_df, False)


def calc_metrics(y_true, y_pred):
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'f1':        f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall':    recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

def train_epoch(model, loader, optimizer, scheduler, device, grad_clip):
    model.train(); total = 0.0
    for batch in tqdm(loader, desc='Train', leave=False):
        optimizer.zero_grad()
        out = model(input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    labels=batch['labels'].to(device))
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step(); scheduler.step()
        total += out.loss.item()
    return total / max(len(loader), 1)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval(); total, y_true, y_pred = 0.0, [], []
    for batch in tqdm(loader, desc='Eval', leave=False):
        out = model(input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    labels=batch['labels'].to(device))
        total += out.loss.item()
        y_true.extend(batch['labels'].numpy())
        y_pred.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
    m = calc_metrics(y_true, y_pred)
    m.update({'loss': total / max(len(loader), 1), 'y_true': y_true, 'y_pred': y_pred})
    return m

def train_rubert(config: Config):
    set_seed(config.random_state); ensure_dir(config.output_dir)
    print(f'Устройство: {config.device}')
    train_df, val_df, test_df = load_dataset(config)
    print(f'Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}')
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        config.model_name, num_labels=2).to(config.device)
    train_loader, val_loader, test_loader = create_dataloaders(
        train_df, val_df, test_df, tokenizer, config)
    total_steps = len(train_loader) * config.epochs
    optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * config.warmup_ratio), total_steps)
    best_f1, best_model_dir = -1.0, os.path.join(config.output_dir, 'best_model')
    for epoch in range(1, config.epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler,
                                  config.device, config.grad_clip)
        val_m = evaluate(model, val_loader, config.device)
        print(f'Epoch {epoch}: train_loss={train_loss:.4f} val_f1={val_m["f1"]:.4f}')
        if val_m['f1'] > best_f1:
            best_f1 = val_m['f1']
            ensure_dir(best_model_dir)
            model.save_pretrained(best_model_dir)
            tokenizer.save_pretrained(best_model_dir)
    best_model = AutoModelForSequenceClassification.from_pretrained(
        best_model_dir).to(config.device)
    test_m = evaluate(best_model, test_loader, config.device)
    print(classification_report(test_m['y_true'], test_m['y_pred'], digits=4))
    save_json(os.path.join(config.output_dir, 'config.json'), asdict(config))
    return best_model, tokenizer

# Раскомментируйте для обучения с нуля:
# config = Config()
# train_rubert(config)
print('Функции обучения определены')


## 5. Инференс RuBERT

In [4]:
def load_rubert(model_dir: str, device: str = None):
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    abs_dir = os.path.abspath(model_dir)   # важно: абсолютный путь, иначе Windows-ошибка
    tokenizer = AutoTokenizer.from_pretrained(abs_dir)
    model = AutoModelForSequenceClassification.from_pretrained(abs_dir).to(device)
    model.eval()
    return tokenizer, model, device

@torch.no_grad()
def predict_text(text, tokenizer, model, device, max_length=256):
    enc = tokenizer(str(text).strip(), truncation=True, padding='max_length',
                    max_length=max_length, return_tensors='pt')
    probs = torch.softmax(
        model(input_ids=enc['input_ids'].to(device),
              attention_mask=enc['attention_mask'].to(device)).logits, dim=1
    ).cpu().numpy()[0]
    pred = int(np.argmax(probs))
    return {'label': pred, 'label_name': 'real' if pred == 1 else 'fake',
            'fake_proba': float(probs[0]), 'real_proba': float(probs[1])}

print('Функции инференса определены')


Функции инференса определены


## 6. RealFactChecker — Evidence Retrieval + NLI

### Схема работы

```
1. Natasha NER → список сущностей из статьи

2. Evidence Retrieval (для каждой сущности):
   ├── Wikipedia API  → предложения из статьи Wikipedia
   └── DuckDuckGo     → сниппеты из топ-3 результатов поиска

3. Для каждого предложения статьи:
   ├── SentenceTransformer → топ-K ближайших доказательных предложений
   └── RuBERT-NLI (threeway) для каждой пары (evidence, claim):
         premise    = доказательство из Wikipedia/DDG
         hypothesis = утверждение из статьи
         → P(entailment), P(neutral), P(contradiction)
         nli_score  = P(entailment) - P(contradiction) ∈ [-1, 1]

4. factcheck_score = 0.5 + 0.5 × mean_nli_score ∈ [0, 1]
   > 0.5 → факты подтверждены источниками
   < 0.5 → факты противоречат источникам
   = 0.5 → нейтрально / нет данных
```


In [5]:
class RealFactChecker:
    """
    Проверяет факты статьи через Wikipedia + DuckDuckGo с использованием NLI.

    Источники:
    - ru.wikipedia.org  (свободный API, доступен в России)
    - DuckDuckGo Search (бесплатно, без API-ключа, доступен в России)

    NLI-модель: cointegrated/rubert-base-cased-nli-threeway
    """

    def __init__(self, config: Config):
        self.config = config
        ensure_dir(config.fc_cache_dir)
        self._cache_path = os.path.join(config.fc_cache_dir, 'evidence_cache.json')
        self._cache = self._load_cache()

        # Natasha NER
        self._segmenter  = Segmenter()
        _emb             = NewsEmbedding()
        self._ner_tagger = NewsNERTagger(_emb)

        # Retrieval model (косинусное сходство для поиска доказательств)
        print('Загружаем retrieval модель...')
        self._retrieval = SentenceTransformer(config.retrieval_model)

        # NLI модель (реальная проверка фактов)
        print('Загружаем NLI модель (cointegrated/rubert-base-cased-nli-threeway)...')
        self._nli_tokenizer = AutoTokenizer.from_pretrained(config.nli_model_name)
        self._nli_model = AutoModelForSequenceClassification.from_pretrained(
            config.nli_model_name).to(config.device)
        self._nli_model.eval()

        # Индексы меток (из model.config.id2label)
        id2label = self._nli_model.config.id2label
        self._entail_idx = next(k for k, v in id2label.items() if 'entail' in v.lower())
        self._neutral_idx = next(k for k, v in id2label.items() if 'neutral' in v.lower())
        self._contra_idx  = next(k for k, v in id2label.items() if 'contra' in v.lower())
        print(f'NLI labels: {id2label}')
        print('RealFactChecker готов')

    # ── Cache ──────────────────────────────────────────────────────────────
    def _load_cache(self):
        if os.path.exists(self._cache_path):
            with open(self._cache_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        return {}

    def _save_cache(self):
        with open(self._cache_path, 'w', encoding='utf-8') as f:
            json.dump(self._cache, f, ensure_ascii=False)

    # ── NER ────────────────────────────────────────────────────────────────
    def extract_entities(self, text: str) -> list:
        doc = Doc(str(text)[:3000])
        doc.segment(self._segmenter)
        doc.tag_ner(self._ner_tagger)
        return list({span.text for span in doc.spans if len(span.text) > 2}
                    )[:self.config.fc_max_entities]

    # ── Evidence Retrieval ─────────────────────────────────────────────────
    def _fetch_wikipedia(self, entity: str) -> list[str]:
        """Возвращает список предложений из Wikipedia-статьи по сущности."""
        key = 'wiki_' + hashlib.md5(entity.encode()).hexdigest()
        if key in self._cache:
            return self._cache[key] or []
        try:
            wikipedia.set_lang('ru')
            results = wikipedia.search(entity, results=2)
            if not results:
                self._cache[key] = []
                return []
            page = wikipedia.page(results[0], auto_suggest=False)
            content = page.content[:self.config.fc_max_wiki_chars]
            sentences = [s.strip() for s in content.split('.') if len(s.strip()) > 25]
            self._cache[key] = sentences
            self._save_cache()
            time.sleep(0.3)
            return sentences
        except Exception:
            self._cache[key] = []
            return []

    def _fetch_duckduckgo(self, query: str) -> list[str]:
        """Возвращает список сниппетов из DuckDuckGo по запросу (без API-ключа)."""
        key = 'ddg_' + hashlib.md5(query.encode()).hexdigest()
        if key in self._cache:
            return self._cache[key] or []
        try:
            sentences = []
            with DDGS() as ddgs:
                results = list(ddgs.text(
                    query, region='ru-ru',
                    max_results=self.config.fc_ddg_results
                ))
            for r in results:
                body = r.get('body', '')
                # Разбиваем сниппет на предложения
                for s in body.split('.'):
                    s = s.strip()
                    if len(s) > 25:
                        sentences.append(s)
            self._cache[key] = sentences
            self._save_cache()
            time.sleep(0.5)
            return sentences
        except Exception:
            self._cache[key] = []
            return []

    def gather_evidence(self, text: str, entities: list) -> list[str]:
        """Собирает доказательства из Wikipedia и DuckDuckGo."""
        evidence = []
        for entity in entities:
            evidence.extend(self._fetch_wikipedia(entity))
            # Дополнительный DDG-запрос: сущность + ключевые слова статьи
            short_query = entity + ' ' + ' '.join(text.split()[:5])
            evidence.extend(self._fetch_duckduckgo(short_query))
        # Удаляем дубликаты, сохраняем порядок
        seen = set()
        unique = []
        for s in evidence:
            if s not in seen:
                seen.add(s); unique.append(s)
        return unique

    # ── NLI ────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def _nli_score(self, premise: str, hypothesis: str) -> float:
        """
        Возвращает NLI-score = P(entailment) - P(contradiction) ∈ [-1, 1].
        Позитивное значение → доказательство поддерживает утверждение.
        Негативное значение → доказательство противоречит утверждению.
        """
        inputs = self._nli_tokenizer(
            premise, hypothesis,
            truncation=True, max_length=512,
            return_tensors='pt'
        ).to(self.config.device)
        probs = torch.softmax(
            self._nli_model(**inputs).logits, dim=-1
        ).cpu().numpy()[0]
        return float(probs[self._entail_idx] - probs[self._contra_idx])

    def _find_top_evidence(self, claim: str, evidence_list: list, top_k: int) -> list[str]:
        """Находит top_k наиболее семантически релевантных доказательств для утверждения."""
        if not evidence_list:
            return []
        claim_emb = self._retrieval.encode([claim], show_progress_bar=False)
        evid_embs = self._retrieval.encode(evidence_list, show_progress_bar=False)
        sims = cos_sim_matrix(claim_emb, evid_embs)[0]
        top_indices = np.argsort(sims)[::-1][:top_k]
        return [evidence_list[i] for i in top_indices]

    # ── Main ───────────────────────────────────────────────────────────────
    def get_score(self, text: str) -> dict:
        """
        Основной метод. Возвращает factcheck_score ∈ [0, 1].
        1.0 = факты хорошо подтверждены источниками
        0.5 = нейтрально / нет данных
        0.0 = факты противоречат источникам
        """
        entities = self.extract_entities(text)
        claims   = [s.strip() for s in text.split('.')
                    if len(s.strip()) > 20][:self.config.fc_max_article_sents]

        if not entities or not claims:
            return {'factcheck_score': 0.5, 'entities': entities,
                    'nli_details': [], 'evidence_count': 0}

        evidence = self.gather_evidence(text, entities)
        if not evidence:
            return {'factcheck_score': 0.5, 'entities': entities,
                    'nli_details': [], 'evidence_count': 0}

        nli_scores, nli_details = [], []
        for claim in claims:
            top_ev = self._find_top_evidence(claim, evidence, self.config.fc_top_k_evidence)
            claim_scores = []
            for ev in top_ev:
                score = self._nli_score(premise=ev, hypothesis=claim)
                claim_scores.append(score)
                nli_details.append({
                    'claim':     claim[:80],
                    'evidence':  ev[:80],
                    'nli_score': round(score, 4),
                })
            if claim_scores:
                # Берём максимальный NLI-score: если хоть одно доказательство
                # подтверждает — считаем утверждение подтверждённым
                nli_scores.append(max(claim_scores))

        mean_nli = float(np.mean(nli_scores)) if nli_scores else 0.0
        # Нормализуем [-1, 1] → [0, 1]
        factcheck_score = 0.5 + 0.5 * mean_nli

        return {
            'factcheck_score': round(factcheck_score, 4),
            'entities':        entities,
            'evidence_count':  len(evidence),
            'mean_nli_score':  round(mean_nli, 4),
            'nli_details':     nli_details,
        }

print('Класс RealFactChecker определён')


Класс RealFactChecker определён


## 7. HybridDetector

In [6]:
class HybridDetector:
    """
    Объединяет RuBERT и RealFactChecker.
    alpha=0.7 → 70% RuBERT + 30% NLI-FactCheck
    """

    def __init__(self, config: Config):
        self.config = config
        print('Загружаем RuBERT...')
        self.tokenizer, self.model, self.device = load_rubert(
            os.path.abspath(os.path.join(config.output_dir, 'best_model')),
            config.device
        )
        print('Загружаем RealFactChecker...')
        self.fact_checker = RealFactChecker(config)
        print(f'HybridDetector готов  (alpha={config.alpha})')

    def predict(self, text: str, use_factcheck: bool = True) -> dict:
        # 1. RuBERT
        rb = predict_text(text, self.tokenizer, self.model,
                           self.device, self.config.max_length)
        if not use_factcheck:
            return {**rb, 'factcheck_score': None,
                    'combined_real_proba': rb['real_proba']}

        # 2. NLI FactCheck
        fc = self.fact_checker.get_score(text)

        # 3. Взвешенное объединение
        a = self.config.alpha
        combined_real = a * rb['real_proba'] + (1 - a) * fc['factcheck_score']
        final_label   = 1 if combined_real >= self.config.threshold else 0

        return {
            'label':               final_label,
            'label_name':          'real' if final_label == 1 else 'fake',
            'rubert_real_proba':   round(rb['real_proba'], 4),
            'rubert_fake_proba':   round(rb['fake_proba'], 4),
            'factcheck_score':     fc['factcheck_score'],
            'mean_nli_score':      fc.get('mean_nli_score', 0.0),
            'combined_real_proba': round(combined_real, 4),
            'combined_fake_proba': round(1 - combined_real, 4),
            'entities':            fc['entities'],
            'evidence_count':      fc['evidence_count'],
            'nli_details':         fc['nli_details'],
        }

    def predict_batch(self, texts: list, use_factcheck: bool = True) -> list:
        return [self.predict(t, use_factcheck)
                for t in tqdm(texts, desc='HybridDetector')]

print('Класс HybridDetector определён')


Класс HybridDetector определён


## 8. Инициализация

In [7]:
config   = Config()
detector = HybridDetector(config)

Загружаем RuBERT...
Загружаем RealFactChecker...
Загружаем retrieval модель...
Загружаем NLI модель (cointegrated/rubert-base-cased-nli-threeway)...
NLI labels: {0: 'entailment', 1: 'contradiction', 2: 'neutral'}
RealFactChecker готов
HybridDetector готов  (alpha=0.7)


## 9. Пример использования

In [8]:
def show_result(result: dict):
    print(f"Итог:              {result['label_name'].upper()} (label={result['label']})")
    print(f"RuBERT real:       {result['rubert_real_proba']:.4f}")
    print(f"FactCheck score:   {result['factcheck_score']}  (NLI mean={result.get('mean_nli_score', '—')})")
    print(f"Combined real:     {result['combined_real_proba']:.4f}")
    print(f"Сущности:          {result['entities']}")
    print(f"Доказательств:     {result['evidence_count']}")
    if result.get('nli_details'):
        print('Топ NLI-пары:')
        top3 = sorted(result['nli_details'], key=lambda x: abs(x['nli_score']), reverse=True)[:3]
        for d in top3:
            sign = '✓' if d['nli_score'] > 0.1 else ('✗' if d['nli_score'] < -0.1 else '~')
            print(f'  [{sign} {d["nli_score"]:+.3f}] claim: {d["claim"][:60]}')
            print(f'           evid:  {d["evidence"][:60]}')

# ── Реальная новость ──────────────────────────────────────────────────────────
real_text = """
Президент России Владимир Путин провёл встречу с членами Совета Безопасности
в Кремле. На встрече обсуждались вопросы национальной безопасности страны.
Пресс-служба Кремля подтвердила информацию официальным сообщением.
"""
print('=== Реальная новость ===')
show_result(detector.predict(real_text))

print()

# ── Фейк ─────────────────────────────────────────────────────────────────────
fake_text = """
Учёные обнаружили, что прививки от гриппа содержат микрочипы,
которые активируются сигналом 5G и влияют на мозговую активность.
Правительства нескольких стран уже признали этот факт на закрытых заседаниях.
"""
print('=== Фейк ===')
show_result(detector.predict(fake_text))


=== Реальная новость ===


C:\Users\pozoy\AppData\Local\Temp\ipykernel_22888\1704504319.py:91: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\pozoy\AppData\Local\Temp\ipykernel_22888\1704504319.py:91: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\pozoy\AppData\Local\Temp\ipykernel_22888\1704504319.py:91: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\pozoy\AppData\Local\Temp\ipykernel_22888\1704504319.py:91: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Итог:              FAKE (label=0)
RuBERT real:       0.0304
FactCheck score:   0.4951  (NLI mean=-0.0099)
Combined real:     0.1698
Сущности:          ['Совета Безопасности', 'Кремле', 'России', 'Владимир Путин']
Доказательств:     168
Топ NLI-пары:
  [✗ -0.662] claim: Президент России Владимир Путин провёл встречу с членами Сов
           evid:  Заместитель председателя Совета безопасности Российской Феде
  [~ -0.081] claim: На встрече обсуждались вопросы национальной безопасности стр
           evid:  Политическая роль Совета безопасности (СБ) в различные этапы
  [~ -0.022] claim: Пресс-служба Кремля подтвердила информацию официальным сообщ
           evid:  == Статус и полномочия ==

Указом президента Российской Феде

=== Фейк ===
Итог:              FAKE (label=0)
RuBERT real:       0.1082
FactCheck score:   0.5  (NLI mean=0.0)
Combined real:     0.2257
Сущности:          []
Доказательств:     0


## 10. Сравнение на тестовой выборке

In [9]:
def evaluate_hybrid(detector, n_samples=50, random_state=42):
    """Сравнивает RuBERT и HybridDetector на n_samples примерах."""
    _, _, test_df = load_dataset(detector.config)
    sample = test_df.sample(min(n_samples, len(test_df)),
                            random_state=random_state).reset_index(drop=True)
    y_true = sample[detector.config.label_column].tolist()
    texts  = sample[detector.config.text_column].tolist()

    preds_rb = [detector.predict(t, use_factcheck=False)['label']
                for t in tqdm(texts, desc='RuBERT only')]

    results_h = detector.predict_batch(texts, use_factcheck=True)
    preds_h   = [r['label'] for r in results_h]

    def m(y_t, y_p):
        return {'accuracy':  round(accuracy_score(y_t, y_p), 4),
                'f1':        round(f1_score(y_t, y_p, pos_label=1, zero_division=0), 4),
                'precision': round(precision_score(y_t, y_p, pos_label=1, zero_division=0), 4),
                'recall':    round(recall_score(y_t, y_p, pos_label=1, zero_division=0), 4)}

    cmp = pd.DataFrame({'RuBERT': m(y_true, preds_rb),
                        'RuBERT + NLI FactCheck': m(y_true, preds_h)}).T
    print(cmp)

    # Анализ расхождений (где модели не совпали)
    log = pd.DataFrame({
        'text':           [t[:100] for t in texts],
        'true':           y_true,
        'rubert_pred':    preds_rb,
        'hybrid_pred':    preds_h,
        'rubert_real':    [r['rubert_real_proba'] for r in results_h],
        'factcheck_score':[r['factcheck_score']   for r in results_h],
        'mean_nli':       [r.get('mean_nli_score', 0) for r in results_h],
        'combined_real':  [r['combined_real_proba'] for r in results_h],
        'entities':       [', '.join(r['entities']) for r in results_h],
    })
    ensure_dir('assets')
    log.to_csv('assets/nli_eval_log.csv', index=False, encoding='utf-8')
    print('Лог → assets/nli_eval_log.csv')
    return cmp

# Запуск: ~10-20 мин из-за Wikipedia + DDG + NLI
# evaluate_hybrid(detector, n_samples=50)


## 11. Подбор alpha

In [10]:
def tune_alpha(detector, alphas=None, n_samples=50):
    if alphas is None:
        alphas = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    _, _, test_df = load_dataset(detector.config)
    sample = test_df.sample(min(n_samples, len(test_df)),
                            random_state=42).reset_index(drop=True)
    y_true = sample[detector.config.label_column].tolist()

    raw = detector.predict_batch(sample[detector.config.text_column].tolist())

    rows = []
    for alpha in alphas:
        preds = []
        for r in raw:
            cr = alpha * r['rubert_real_proba'] + (1 - alpha) * r['factcheck_score']
            preds.append(1 if cr >= detector.config.threshold else 0)
        rows.append({'alpha': alpha,
                     'accuracy': round(accuracy_score(y_true, preds), 4),
                     'f1': round(f1_score(y_true, preds, pos_label=1, zero_division=0), 4)})

    df = pd.DataFrame(rows)
    best = df.loc[df['f1'].idxmax()]
    print(df.to_string(index=False))
    print(f'\nЛучший alpha = {best["alpha"]}  (F1 = {best["f1"]})')
    return df

# tune_alpha(detector, n_samples=50)


## Замечания

- **Скорость**: Wikipedia (~0.5 с) + DDG (~1 с) + NLI (per pair) ≈ 5–15 сек/статья.
  Кэш `fact_check_cache/evidence_cache.json` снижает задержку при повторных запросах.
- **NLI-интерпретация**: `nli_score > 0.1` → факт подтверждён; `< -0.1` → противоречие; между ними → нейтрально.
- **Ограничения**: NLI работает на коротких парах. Очень длинные предложения truncate до 512 токенов. Для лучшего качества разбивай длинные предложения на подпредложения.
- **Источники**: Wikipedia и DuckDuckGo доступны в России без VPN и без API-ключей.
